In [ ]:
!pip install -q transformers accelerate bitsandbytes seaborn scikit-learn pandas

In [ ]:
# ============================================================
# PropInsight — QLoRA FINETUNE (h2oai/danube2-singlish-finetuned)
# Uses existing /propinsight_datasets/{train,validation,test}.json
# Saves LoRA adapters to /models/finetuned/_Danube2
# Saves predictions/metrics to /results/_Danube2
# ============================================================
# ============================================================
# PropInsight — Danube2 Singlish: BASELINE + QLoRA FINETUNE
# Splits: /content/drive/MyDrive/PropInsight/propinsight_datasets/{train,validation,test}.json
# Outputs under: results/_Danube2, visualizations/_Danube2, models/finetuned/_Danube2
# ============================================================

# --------- CONFIG ----------
#RUN_BASELINE = True     # set False to skip baseline
#RUN_FINETUNE = True     # set False to skip finetune
#EPOCHS       = 3        # finetune epochs (suggest 2–3)
#MAX_LEN      = 2048
#MODEL_ID     = "h2oai/danube2-singlish-finetuned"
## --------------------------
#
## 1) Imports
#import os, json, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, torch
#from pathlib import Path
#from tqdm.auto import tqdm
#from collections import Counter
#from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
#from transformers import (
#    AutoTokenizer, AutoModelForCausalLM,
#    BitsAndBytesConfig, TrainingArguments, Trainer, EarlyStoppingCallback
#)
#from datasets import Dataset
#from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
#from google.colab import drive
#
## 2) Drive + Dirs
#drive.mount('/content/drive')
#BASE = Path("/content/drive/MyDrive/PropInsight")
#
#DATASETS_DIR = BASE / "propinsight_datasets"
#TRAIN_JSON   = DATASETS_DIR / "train.json"
#VAL_JSON     = DATASETS_DIR / "validation.json"
#TEST_JSON    = DATASETS_DIR / "test.json"
#
#RESULTS_DIR        = BASE / "results"
#VISUALIZATIONS_DIR = BASE / "visualizations"
#MODELS_DIR         = BASE / "models"
#
#DANUBE_TAG = "_Danube2"
#RESULTS_D2 = RESULTS_DIR / DANUBE_TAG
#VIS_D2     = VISUALIZATIONS_DIR / DANUBE_TAG
#FINETUNED_DIR = MODELS_DIR / "finetuned" / DANUBE_TAG
#(RESULTS_D2).mkdir(parents=True, exist_ok=True)
#(VIS_D2).mkdir(parents=True, exist_ok=True)
#(FINETUNED_DIR).mkdir(parents=True, exist_ok=True)
#
#print("✅ Dirs ready")
#print(f"Splits present? train={TRAIN_JSON.exists()} val={VAL_JSON.exists()} test={TEST_JSON.exists()}")
#
#if not (TRAIN_JSON.exists() and VAL_JSON.exists() and TEST_JSON.exists()):
#    raise FileNotFoundError("Missing split(s). Ensure train/validation/test.json exist under propinsight_datasets.")
#
#train_data = json.load(open(TRAIN_JSON, "r", encoding="utf-8"))
#val_data   = json.load(open(VAL_JSON,   "r", encoding="utf-8"))
#test_data  = json.load(open(TEST_JSON,  "r", encoding="utf-8"))
#print(f"✓ Loaded: train={len(train_data)} | val={len(val_data)} | test={len(test_data)}")
#
## 3) Tokenizer + (baseline) model loader
#tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)
#if tokenizer.pad_token is None:
#    tokenizer.pad_token = tokenizer.eos_token
#    tokenizer.pad_token_id = tokenizer.eos_token_id
#
#def load_fp_model():
#    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
#    mdl = AutoModelForCausalLM.from_pretrained(
#        MODEL_ID, torch_dtype=dtype, device_map="auto", trust_remote_code=True
#    )
#    mdl.eval()
#    return mdl
#
## 4) Danube2 prompt helpers
#SYSTEM_MSG = (
#    "You are PropInsight, a Singapore real-estate expert. "
#    "Return ONLY the Output block with fields: Overall Sentiment, Price Sentiment, "
#    "Policy Sentiment, Affordability Sentiment, Location, Aspect, Entity, Policy Mentioned, "
#    "Singlish Detected, Cultural Context, Emotion, Datetime, Source, Reasoning."
#)
#
#def build_prompt_user(text):
#    return (
#        f"{SYSTEM_MSG}\n\n"
#        f"Analyze the sentiment of this Singapore property comment:\n\n"
#        f"{text}\n\n"
#        f"Return only the Output block."
#    )
#
#def prompt_for_infer(text):
#    return f"<|prompt|>{build_prompt_user(text)}</s><|answer|>"
#
#def extract_field(block: str, key: str, default: str = "") -> str:
#    m = re.search(rf"{re.escape(key)}\s*:\s*(.*)", block or "", flags=re.IGNORECASE)
#    return (m.group(1).strip() if m else default)
#
#def pick_overall(block: str) -> str:
#    val = extract_field(block, "Overall Sentiment", default="neutral").lower()
#    return val if val else "neutral"
#
## 5) BASELINE (no training)
#if RUN_BASELINE:
#    print("\n=== BASELINE (Danube2) ===")
#    model_baseline = load_fp_model()
#    preds = []
#    for item in tqdm(test_data, total=len(test_data), desc="Baseline inference"):
#        inp = item.get("input", "")
#        if not isinstance(inp, str) or not inp.strip():
#            preds.append({"input": inp, "ground_truth": item.get("output",""), "prediction": "", "metadata": item.get("metadata", {})})
#            continue
#        prompt = prompt_for_infer(inp)
#        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model_baseline.device)
#        with torch.no_grad():
#            out = model_baseline.generate(
#                **inputs, max_new_tokens=512, temperature=0.7, top_p=0.9,
#                repetition_penalty=1.1, do_sample=True
#            )
#        text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
#        preds.append({"input": inp, "ground_truth": item.get("output",""), "prediction": text, "metadata": item.get("metadata", {})})
#
#    baseline_path = RESULTS_D2 / "baseline_predictions.json"
#    json.dump(preds, open(baseline_path, "w"), indent=2, ensure_ascii=False)
#    print(f"✓ Saved baseline predictions → {baseline_path}")
#
#    y_true = [pick_overall(p["ground_truth"]) for p in preds]
#    y_pred = [pick_overall(p["prediction"])   for p in preds]
#    gt_counts = Counter(y_true); pd_counts = Counter(y_pred)
#    labels = sorted(set(list(gt_counts.keys()) + list(pd_counts.keys())))
#    overall_acc = np.mean([t == p for t, p in zip(y_true, y_pred)]) if len(y_true) else 0.0
#    json.dump({
#        "samples": len(preds),
#        "overall_sentiment_accuracy": float(overall_acc),
#        "label_set": labels,
#        "ground_truth_distribution": dict(gt_counts),
#        "prediction_distribution": dict(pd_counts),
#    }, open(RESULTS_D2 / "baseline_metrics.json", "w"), indent=2)
#    print(f"✓ Baseline accuracy: {overall_acc:.3f}")
#
#    sns.set_style("whitegrid")
#    x = np.arange(len(labels))
#    gt_vals = [gt_counts.get(l,0) for l in labels]
#    pd_vals = [pd_counts.get(l,0) for l in labels]
#    fig, ax = plt.subplots(figsize=(11,5))
#    ax.bar(x-0.35/2, gt_vals, width=0.35, label="Ground Truth")
#    ax.bar(x+0.35/2, pd_vals, width=0.35, label="Predicted")
#    ax.set_xticks(x); ax.set_xticklabels(labels)
#    ax.set_title("Overall Sentiment — Baseline (Danube2)")
#    ax.legend(); ax.grid(axis="y", alpha=.3)
#    plt.tight_layout(); plt.savefig(VIS_D2 / "baseline_sentiment_comparison.png", dpi=300); plt.close()
#
#    cm = confusion_matrix(y_true, y_pred, labels=labels)
#    with np.errstate(invalid='ignore'):
#        cmn = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
#    fig, ax = plt.subplots(figsize=(8,6))
#    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues",
#                xticklabels=labels, yticklabels=labels, ax=ax,
#                cbar_kws={"label":"Proportion"})
#    ax.set_xlabel("Predicted"); ax.set_ylabel("Ground Truth")
#    ax.set_title("Baseline Confusion Matrix (Danube2)")
#    plt.tight_layout(); plt.savefig(VIS_D2 / "baseline_confusion_matrix.png", dpi=300); plt.close()
#    print("✅ BASELINE COMPLETE")
#
## 6) QLoRA FINETUNE
#if RUN_FINETUNE:
#    print("\n=== QLoRA FINETUNE (Danube2) ===")
#
#    # Build SFT datasets in Danube2 format
#    def sft_text(sample: dict) -> str:
#        instr = sample.get("instruction","Analyze the sentiment of this Singapore property comment:")
#        text  = sample.get("input","")
#        out   = sample.get("output","")
#        user  = f"{SYSTEM_MSG}\n\n{instr}\n\nInput: {text}\n\nReturn only the Output block."
#        return f"<|prompt|>{user}</s><|answer|>{out}"
#
#    train_ds = Dataset.from_list([{"text": sft_text(x)} for x in train_data])
#    val_ds   = Dataset.from_list([{"text": sft_text(x)} for x in val_data])
#
#    def tok_fn(batch):
#        t = tokenizer(batch["text"], max_length=MAX_LEN, truncation=True, padding="max_length")
#        t["labels"] = t["input_ids"].copy()
#        return t
#
#    train_ds = train_ds.map(tok_fn, batched=True, remove_columns=["text"])
#    val_ds   = val_ds.map(tok_fn,   batched=True, remove_columns=["text"])
#
#    # Load 4-bit base + LoRA
#    bnb_config = BitsAndBytesConfig(
#        load_in_4bit=True,
#        bnb_4bit_quant_type="nf4",
#        bnb_4bit_compute_dtype=torch.bfloat16,
#        bnb_4bit_use_double_quant=True,
#    )
#    base_model = AutoModelForCausalLM.from_pretrained(
#        MODEL_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
#    )
#    base_model.config.use_cache = False
#    base_model = prepare_model_for_kbit_training(base_model)
#
#    lora_cfg = LoraConfig(
#        r=16, lora_alpha=32, lora_dropout=0.05,
#        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
#        task_type="CAUSAL_LM"
#    )
#    model = get_peft_model(base_model, lora_cfg)
#    model.print_trainable_parameters()
#
#    bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
#    args = TrainingArguments(
#        output_dir=str(FINETUNED_DIR),
#        per_device_train_batch_size=1,
#        per_device_eval_batch_size=1,
#        gradient_accumulation_steps=16,
#        num_train_epochs=EPOCHS,
#        learning_rate=2e-4,
#        warmup_ratio=0.03,
#        weight_decay=0.01,
#
#        eval_strategy="steps",
#        eval_steps=50,
#        save_strategy="steps",
#        save_steps=50,
#        save_total_limit=3,
#
#        logging_steps=5,
#        logging_dir=str(RESULTS_D2 / "tensorboard"),
#        report_to=["tensorboard"],
#
#        bf16=bf16_ok,
#        fp16=not bf16_ok,
#
#        optim="paged_adamw_8bit",
#        lr_scheduler_type="cosine",
#        gradient_checkpointing=True,
#        remove_unused_columns=False,
#
#        load_best_model_at_end=True,
#        metric_for_best_model="eval_loss",
#        greater_is_better=False,
#    )
#    early_stop = EarlyStoppingCallback(early_stopping_patience=3)
#
#    # Save training config
#    json.dump({
#        "model_id": MODEL_ID,
#        "bnb_4bit": {"quant_type":"nf4","compute_dtype":"bfloat16","double_quant":True},
#        "lora": {"r":16,"alpha":32,"dropout":0.05,"targets":["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]},
#        "epochs": args.num_train_epochs,
#        "batch_size": args.per_device_train_batch_size,
#        "grad_accum": args.gradient_accumulation_steps,
#        "lr": args.learning_rate,
#        "max_len": MAX_LEN
#    }, open(RESULTS_D2 / "training_config.json", "w"), indent=2)
#
#    trainer = Trainer(
#        model=model, args=args,
#        train_dataset=train_ds, eval_dataset=val_ds,
#        callbacks=[early_stop],
#        tokenizer=tokenizer
#    )
#
#    trainer.train()
#    trainer.save_model(str(FINETUNED_DIR))
#    tokenizer.save_pretrained(str(FINETUNED_DIR))
#    print(f"✓ Saved LoRA adapters → {FINETUNED_DIR}")
#
#    # Inference with finetuned adapters
#    def gen_output(mdl, tok, text: str) -> str:
#        prompt = prompt_for_infer(text)
#        inp = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(mdl.device)
#        with torch.no_grad():
#            out = mdl.generate(
#                **inp, max_new_tokens=512, temperature=0.7, top_p=0.9,
#                repetition_penalty=1.1, do_sample=True
#            )
#        return tok.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True)
#
#    finetuned_preds = []
#    for item in tqdm(test_data, total=len(test_data), desc="Finetuned inference"):
#        txt = item.get("input","")
#        pred = gen_output(model, tokenizer, txt) if isinstance(txt, str) and txt.strip() else ""
#        finetuned_preds.append({
#            "input": txt,
#            "ground_truth": item.get("output",""),
#            "prediction": pred,
#            "metadata": item.get("metadata", {})
#        })
#
#    FINETUNED_PREDICTIONS = RESULTS_D2 / "finetuned_predictions.json"
#    json.dump(finetuned_preds, open(FINETUNED_PREDICTIONS, "w"), indent=2, ensure_ascii=False)
#    print(f"✓ Saved predictions → {FINETUNED_PREDICTIONS}")
#
#    def ext(block, key, default="neutral"):
#        m = re.search(rf"{re.escape(key)}\s*:\s*(.*)", block or "", re.IGNORECASE)
#        return (m.group(1).strip().lower() if m else default)
#
#    y_true = [ext(p["ground_truth"], "Overall Sentiment") for p in finetuned_preds]
#    y_pred = [ext(p["prediction"],   "Overall Sentiment") for p in finetuned_preds]
#    labels = sorted(set(y_true) | set(y_pred))
#    metrics = {
#        "samples": len(finetuned_preds),
#        "overall_sentiment_accuracy": float(accuracy_score(y_true, y_pred)) if len(y_true) else 0.0,
#        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
#        "macro_precision": float(precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
#        "macro_recall": float(recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
#        "labels": labels,
#    }
#    json.dump(metrics, open(RESULTS_D2 / "evaluation_results.json", "w"), indent=2)
#    print("✓ Finetuned-only metrics saved:", RESULTS_D2 / "evaluation_results.json")
#    print("✅ FINETUNE COMPLETE")
#
#print("\nAll done.")
#
#

In [ ]:
# ============================================================
# PropInsight — Danube2 Singlish: BASELINE + QLoRA FINETUNE
# QUICK TEST MODE for rapid verification
# ============================================================

# --------- CONFIG ----------
QUICK_TEST   = False     # 🚀 SET TO True FOR QUICK TEST, False FOR FULL RUN
RUN_BASELINE = True     # set False to skip baseline
RUN_FINETUNE = True     # set False to skip finetune

# Quick test settings (used when QUICK_TEST=True)
if QUICK_TEST:
    TRAIN_SAMPLES = 50      # Use only 50 training samples
    VAL_SAMPLES   = 20      # Use only 20 validation samples
    TEST_SAMPLES  = 30      # Use only 30 test samples
    EPOCHS        = 1       # Just 1 epoch
    MAX_LEN       = 512     # Shorter sequences
    EVAL_STEPS    = 10      # Evaluate more frequently
    SAVE_STEPS    = 10      # Save more frequently
    print("🚀 QUICK TEST MODE ENABLED")
    print(f"   Train: {TRAIN_SAMPLES} | Val: {VAL_SAMPLES} | Test: {TEST_SAMPLES}")
    print(f"   Epochs: {EPOCHS} | Max Length: {MAX_LEN}")
else:
    TRAIN_SAMPLES = None    # Use all data
    VAL_SAMPLES   = None
    TEST_SAMPLES  = None
    EPOCHS        = 3       # Full training
    MAX_LEN       = 2048    # Full length
    EVAL_STEPS    = 50
    SAVE_STEPS    = 50
    print("📊 FULL TRAINING MODE")

MODEL_ID = "h2oai/danube2-singlish-finetuned"
# --------------------------







📊 FULL TRAINING MODE


In [ ]:

import os, json, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, torch
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from google.colab import drive
from IPython.display import display, HTML, Markdown
import warnings
warnings.filterwarnings('ignore')





In [ ]:

# 2) Drive + Dirs
drive.mount('/content/drive')
BASE = Path("/content/drive/MyDrive/PropInsight")

DATASETS_DIR = BASE / "propinsight_datasets"
TRAIN_JSON   = DATASETS_DIR / "train.json"
VAL_JSON     = DATASETS_DIR / "validation.json"
TEST_JSON    = DATASETS_DIR / "test.json"

RESULTS_DIR        = BASE / "results"
VISUALIZATIONS_DIR = BASE / "visualizations"
MODELS_DIR         = BASE / "models"

DANUBE_TAG = "_Danube2_QUICKTEST" if QUICK_TEST else "_Danube2"
RESULTS_D2 = RESULTS_DIR / DANUBE_TAG
VIS_D2     = VISUALIZATIONS_DIR / DANUBE_TAG
FINETUNED_DIR = MODELS_DIR / "finetuned" / DANUBE_TAG
(RESULTS_D2).mkdir(parents=True, exist_ok=True)
(VIS_D2).mkdir(parents=True, exist_ok=True)
(FINETUNED_DIR).mkdir(parents=True, exist_ok=True)

# ========== CORPUS DIRECTORIES ==========
CORPUS_DIR = BASE / "corpus"
SINGLISH_DIR = CORPUS_DIR / "Singlish"  # dictionary/, vocabulary/, lexicon.csv
PROP_DIR = CORPUS_DIR / "SGPropertyDomain"  # glossary.csv, regex_patterns.jsonl


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def load_corpus_resources():
    """Load Singlish lexicon and property domain knowledge"""
    resources = {
        'singlish_lexicon': None,
        'property_glossary': None,
        'regex_patterns': []
    }

    # Load Singlish lexicon - Column is "Word" not "term"!
    singlish_lexicon_path = SINGLISH_DIR / "lexicon.csv"
    if singlish_lexicon_path.exists():
        try:
            df = pd.read_csv(singlish_lexicon_path)

            # Rename "Word" to "term" for consistency
            if 'Word' in df.columns:
                df = df.rename(columns={'Word': 'term'})
                resources['singlish_lexicon'] = df
                print(f"✓ Loaded {len(df)} Singlish terms")
            elif 'term' in df.columns:
                resources['singlish_lexicon'] = df
                print(f"✓ Loaded {len(df)} Singlish terms")
            else:
                print(f"⚠️  Singlish lexicon unexpected columns: {list(df.columns)}")

        except Exception as e:
            print(f"⚠️  Error loading Singlish lexicon: {e}")
    else:
        print(f"⚠️  Singlish lexicon not found at {singlish_lexicon_path}")

    # Load property glossary - Already has "term" column
    glossary_path = PROP_DIR / "glossary.csv"
    if glossary_path.exists():
        try:
            df = pd.read_csv(glossary_path)

            if 'term' in df.columns:
                resources['property_glossary'] = df
                print(f"✓ Loaded {len(df)} property terms")
            else:
                print(f"⚠️  Property glossary 'term' column not found")

        except Exception as e:
            print(f"⚠️  Error loading property glossary: {e}")
    else:
        print(f"⚠️  Property glossary not found at {glossary_path}")

    # Load regex patterns
    regex_path = PROP_DIR / "regex_patterns.jsonl"
    if regex_path.exists():
        try:
            with open(regex_path, 'r') as f:
                for line in f:
                    try:
                        resources['regex_patterns'].append(json.loads(line))
                    except:
                        continue
            print(f"✓ Loaded {len(resources['regex_patterns'])} regex patterns")
        except Exception as e:
            print(f"⚠️  Error loading regex patterns: {e}")
    else:
        print(f"⚠️  Regex patterns not found at {regex_path}")

    return resources

# Load corpus resources
corpus_resources = load_corpus_resources()
print("✅ Corpus loaded\n")
# ========================================

print("✅ Dirs ready")
print(f"Splits present? train={TRAIN_JSON.exists()} val={VAL_JSON.exists()} test={TEST_JSON.exists()}")

if not (TRAIN_JSON.exists() and VAL_JSON.exists() and TEST_JSON.exists()):
    raise FileNotFoundError("Missing split(s). Ensure train/validation/test.json exist under propinsight_datasets.")

# Load data with optional sampling
train_data = json.load(open(TRAIN_JSON, "r", encoding="utf-8"))
val_data   = json.load(open(VAL_JSON,   "r", encoding="utf-8"))
test_data  = json.load(open(TEST_JSON,  "r", encoding="utf-8"))

if QUICK_TEST:
    import random
    random.seed(42)
    train_data = random.sample(train_data, min(TRAIN_SAMPLES, len(train_data)))
    val_data   = random.sample(val_data,   min(VAL_SAMPLES,   len(val_data)))
    test_data  = random.sample(test_data,  min(TEST_SAMPLES,  len(test_data)))
    print(f"🎯 Quick test: Sampled {len(train_data)} train | {len(val_data)} val | {len(test_data)} test")
else:
    print(f"✓ Loaded: train={len(train_data)} | val={len(val_data)} | test={len(test_data)}")

# 3) Tokenizer + (baseline) model loader
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

def load_fp_model():
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    mdl = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=dtype, device_map="auto", trust_remote_code=True
    )
    mdl.eval()
    return mdl

✓ Loaded 1783 Singlish terms
✓ Loaded 1025 property terms
✓ Loaded 2256 regex patterns
✅ Corpus loaded

✅ Dirs ready
Splits present? train=True val=True test=True
✓ Loaded: train=2825 | val=314 | test=554


In [ ]:


# 4) Danube2 prompt helpers
SYSTEM_MSG = (
    "You are PropInsight, a Singapore real-estate expert. "
    "Return ONLY the Output block with fields: Overall Sentiment, Price Sentiment, "
    "Policy Sentiment, Affordability Sentiment, Location, Aspect, Entity, Policy Mentioned, "
    "Singlish Detected, Cultural Context, Emotion, Datetime, Source, Reasoning."
)

def build_prompt_user(text):
    return (
        f"{SYSTEM_MSG}\n\n"
        f"Analyze the sentiment of this Singapore property comment:\n\n"
        f"{text}\n\n"
        f"Return only the Output block."
    )

def prompt_for_infer(text):
    return f"<|prompt|>{build_prompt_user(text)}</s><|answer|>"

def extract_field(block: str, key: str, default: str = "") -> str:
    m = re.search(rf"{re.escape(key)}\s*:\s*(.*)", block or "", flags=re.IGNORECASE)
    return (m.group(1).strip() if m else default)

def pick_overall(block: str) -> str:
    val = extract_field(block, "Overall Sentiment", default="neutral").lower()
    return val if val else "neutral"

# ========== DOMAIN-AWARE EVALUATION FUNCTIONS ==========
def check_singlish_coverage(text, prediction, singlish_lexicon):
    """Check if Singlish terms in text are handled correctly"""
    try:
        if singlish_lexicon is None or len(singlish_lexicon) == 0:
            return 1.0

        if 'term' not in singlish_lexicon.columns:
            return 1.0

        # Safe extraction with NaN handling
        singlish_terms = set(
            singlish_lexicon['term']
            .dropna()
            .astype(str)
            .str.lower()
            .str.strip()
        )
        singlish_terms = {t for t in singlish_terms if t}  # Remove empty strings

        text_lower = text.lower()
        found_terms = [term for term in singlish_terms if term in text_lower]

        if not found_terms:
            return 1.0

        pred_lower = prediction.lower()
        singlish_detected = 'singlish' in pred_lower or any(term in pred_lower for term in found_terms)

        return 1.0 if singlish_detected else 0.0

    except Exception as e:
        return 1.0

def check_property_entities(text, prediction, property_glossary, regex_patterns):
    """Check if property domain entities are recognized"""
    try:
        score = 0.0
        count = 0

        # Check glossary terms
        if property_glossary is not None and len(property_glossary) > 0:
            if 'term' in property_glossary.columns:
                # Safe extraction with NaN handling
                prop_terms = set(
                    property_glossary['term']
                    .dropna()
                    .astype(str)
                    .str.lower()
                    .str.strip()
                )
                prop_terms = {t for t in prop_terms if t}  # Remove empty strings

                text_lower = text.lower()
                found_terms = [term for term in prop_terms if term in text_lower]

                if found_terms:
                    count += 1
                    pred_lower = prediction.lower()
                    entities_mentioned = any(term in pred_lower for term in found_terms)
                    score += 1.0 if entities_mentioned else 0.0

        # Check regex patterns
        if regex_patterns and len(regex_patterns) > 0:
            for pattern_obj in regex_patterns:
                pattern = pattern_obj.get('pattern', '')
                if pattern:
                    try:
                        if re.search(pattern, text, re.IGNORECASE):
                            count += 1
                            if re.search(pattern, prediction, re.IGNORECASE):
                                score += 1.0
                    except:
                        continue

        return score / count if count > 0 else 1.0

    except Exception as e:
        return 1.0
def plot_domain_metrics(baseline_metrics, finetuned_metrics, save_path):
    """Visualize domain-specific metrics comparison"""
    fig, ax = plt.subplots(figsize=(12, 6))

    metrics_names = ['Singlish Coverage', 'Entity Recognition', 'Domain Avg']
    baseline_vals = [
        baseline_metrics.get('singlish_coverage', 0),
        baseline_metrics.get('entity_recognition', 0),
        baseline_metrics.get('domain_avg', 0)
    ]
    finetuned_vals = [
        finetuned_metrics.get('singlish_coverage', 0),
        finetuned_metrics.get('entity_recognition', 0),
        finetuned_metrics.get('domain_avg', 0)
    ]

    x = np.arange(len(metrics_names))
    width = 0.35

    bars1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline',
                   color='#e74c3c', alpha=0.8, edgecolor='black', linewidth=1.2)
    bars2 = ax.bar(x + width/2, finetuned_vals, width, label='Finetuned',
                   color='#27ae60', alpha=0.8, edgecolor='black', linewidth=1.2)

    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title('Domain Knowledge Metrics: Baseline vs Finetuned', fontsize=14, fontweight='bold', pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_names, fontsize=11)
    ax.legend(fontsize=11, loc='upper left')
    ax.set_ylim(0, 1.1)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Domain metrics visualization saved to {save_path}")
# ========================================



In [ ]:
if corpus_resources.get('singlish_lexicon') is not None:
    print("Singlish Lexicon Columns:", corpus_resources['singlish_lexicon'].columns.tolist())
else:
    print("Singlish lexicon was not loaded.")

if corpus_resources.get('property_glossary') is not None:
    print("Property Glossary Columns:", corpus_resources['property_glossary'].columns.tolist())
else:
    print("Property glossary was not loaded.")

Singlish Lexicon Columns: ['term', 'Description', 'Description OK', 'Description (updated)', 'Description(Final)', 'Example(Final)', 'Alternate spellings', 'Remarks (if applicable)', 'POS', 'Pronun', 'Origin', 'License', 'Reference', 'Unnamed: 13']
Property Glossary Columns: ['id', 'term', 'category', 'definition', 'aliases']


In [ ]:
 # 5) BASELINE inference
def baseline_inference(test_d):
    mdl = load_fp_model()
    preds = []
    for ex in tqdm(test_d, desc="Baseline inference"):
        inp = ex['input']
        prompt = prompt_for_infer(inp)
        ids = tokenizer.encode(prompt, return_tensors="pt").to(mdl.device)
        out_ids = mdl.generate(
            ids, max_new_tokens=800, do_sample=False,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id
        )
        raw = tokenizer.decode(out_ids[0], skip_special_tokens=True)
        block = raw.split("<|answer|>")[-1].strip() if "<|answer|>" in raw else raw

        # Extract domain metrics
        singlish_score = check_singlish_coverage(
            inp, block, corpus_resources.get('singlish_lexicon')
        )
        entity_score = check_property_entities(
            inp, block,
            corpus_resources.get('property_glossary'),
            corpus_resources.get('regex_patterns')
        )

        preds.append({
            "input": inp,
            "prediction": block,
            "overall_pred": pick_overall(block),
            "ground_truth": ex.get("output", ""),
            "singlish_score": singlish_score,
            "entity_score": entity_score,
        })
    del mdl
    torch.cuda.empty_cache()
    return preds

# 6) Plotting functions
def plot_sentiment_distribution(y_true, y_pred, title, save_path, colors):
    data = pd.DataFrame({
        "Ground Truth": y_true,
        "Predicted": y_pred
    })
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Ground Truth
    data["Ground Truth"].value_counts().sort_index().plot(
        kind='bar', ax=axes[0], color=colors[0], alpha=0.85, edgecolor='black', linewidth=1.2
    )
    axes[0].set_title("Ground Truth Distribution", fontsize=12, fontweight='bold')
    axes[0].set_xlabel("Sentiment", fontsize=11)
    axes[0].set_ylabel("Count", fontsize=11)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(True, alpha=0.3, axis='y', linestyle='--')

    # Predicted
    data["Predicted"].value_counts().sort_index().plot(
        kind='bar', ax=axes[1], color=colors[1], alpha=0.85, edgecolor='black', linewidth=1.2
    )
    axes[1].set_title("Predicted Distribution", fontsize=12, fontweight='bold')
    axes[1].set_xlabel("Sentiment", fontsize=11)
    axes[1].set_ylabel("Count", fontsize=11)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(True, alpha=0.3, axis='y', linestyle='--')

    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_confusion_matrix(y_true, y_pred, labels, title, save_path, cmap):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=labels, yticklabels=labels,
                cbar_kws={'label': 'Count'}, linewidths=0.5, linecolor='gray', ax=ax)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel("Predicted Label", fontsize=12, fontweight='bold')
    ax.set_ylabel("True Label", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def display_metrics_table(base_m, ft_m):
    df = pd.DataFrame({
        "Metric": [
            "Overall Sentiment Accuracy",
            "Macro Precision",
            "Macro Recall",
            "Macro F1",
            "Weighted F1",
            "Singlish Coverage",
            "Entity Recognition",
            "Domain Avg"
        ],
        "Baseline": [
            f"{base_m['overall_sentiment_accuracy']:.4f}",
            f"{base_m['macro_precision']:.4f}",
            f"{base_m['macro_recall']:.4f}",
            f"{base_m['macro_f1']:.4f}",
            f"{base_m['weighted_f1']:.4f}",
            f"{base_m.get('singlish_coverage', 0):.4f}",
            f"{base_m.get('entity_recognition', 0):.4f}",
            f"{base_m.get('domain_avg', 0):.4f}",
        ],
        "Finetuned": [
            f"{ft_m['overall_sentiment_accuracy']:.4f}",
            f"{ft_m['macro_precision']:.4f}",
            f"{ft_m['macro_recall']:.4f}",
            f"{ft_m['macro_f1']:.4f}",
            f"{ft_m['weighted_f1']:.4f}",
            f"{ft_m.get('singlish_coverage', 0):.4f}",
            f"{ft_m.get('entity_recognition', 0):.4f}",
            f"{ft_m.get('domain_avg', 0):.4f}",
        ]
    })

    # Calculate improvements
    improvements = []
    for base_val, ft_val in zip(df["Baseline"], df["Finetuned"]):
        try:
            delta = float(ft_val) - float(base_val)
            improvements.append(f"{delta:+.4f}")
        except:
            improvements.append("N/A")

    df["Δ (Improvement)"] = improvements

    display(HTML(df.to_html(index=False, escape=False, justify='center')))


baseline_metrics = {}
if RUN_BASELINE:
    display(Markdown("---"))
    display(Markdown("# 🔍 BASELINE EVALUATION"))

    baseline_preds = baseline_inference(test_data)
    json.dump(baseline_preds, open(RESULTS_D2 / "baseline_predictions.json", "w"), indent=2)

    y_true_bl = [ex["output"].split("Overall Sentiment:")[-1].split("\n")[0].strip().lower()
                 for ex in test_data]
    y_pred_bl = [p["overall_pred"] for p in baseline_preds]
    labels_bl = sorted(set(y_true_bl + y_pred_bl))

    # Standard metrics
    baseline_metrics = {
        "overall_sentiment_accuracy": float(accuracy_score(y_true_bl, y_pred_bl)),
        "macro_precision": float(precision_score(y_true_bl, y_pred_bl, labels=labels_bl, average='macro', zero_division=0)),
        "macro_recall": float(recall_score(y_true_bl, y_pred_bl, labels=labels_bl, average='macro', zero_division=0)),
        "macro_f1": float(f1_score(y_true_bl, y_pred_bl, labels=labels_bl, average='macro', zero_division=0)),
        "weighted_f1": float(f1_score(y_true_bl, y_pred_bl, labels=labels_bl, average='weighted', zero_division=0)),
    }

    # Domain-specific metrics
    singlish_scores = [p['singlish_score'] for p in baseline_preds]
    entity_scores = [p['entity_score'] for p in baseline_preds]

    baseline_metrics.update({
        "singlish_coverage": float(np.mean(singlish_scores)) if singlish_scores else 0.0,
        "entity_recognition": float(np.mean(entity_scores)) if entity_scores else 0.0,
    })
    baseline_metrics["domain_avg"] = (
        baseline_metrics["singlish_coverage"] + baseline_metrics["entity_recognition"]
    ) / 2.0

    json.dump(baseline_metrics, open(RESULTS_D2 / "baseline_metrics.json", "w"), indent=2)

    plot_sentiment_distribution(y_true_bl, y_pred_bl,
                               "Overall Sentiment Distribution — Baseline (Danube2)",
                               VIS_D2 / "baseline_sentiment_comparison.png",
                               colors=['#e74c3c', '#3498db'])

    plot_confusion_matrix(y_true_bl, y_pred_bl, labels_bl,
                         "Baseline Confusion Matrix (Danube2)",
                         VIS_D2 / "baseline_confusion_matrix.png",
                         cmap='Reds')

    display(Markdown(f"""
    **Baseline Results:**
    - Overall Sentiment Accuracy: **{baseline_metrics['overall_sentiment_accuracy']:.4f}**
    - Macro F1: **{baseline_metrics['macro_f1']:.4f}**
    - Singlish Coverage: **{baseline_metrics['singlish_coverage']:.4f}**
    - Entity Recognition: **{baseline_metrics['entity_recognition']:.4f}**
    - Domain Average: **{baseline_metrics['domain_avg']:.4f}**
    """))

---

# 🔍 BASELINE EVALUATION

Baseline inference:   0%|          | 0/554 [00:00<?, ?it/s]


   **Baseline Results:**
   - Overall Sentiment Accuracy: **0.2942**
   - Macro F1: **0.0333**
   - Singlish Coverage: **1.0000**
   - Entity Recognition: **0.6697**
   - Domain Average: **0.8348**
   

In [ ]:
# 6) QLoRA FINETUNE
finetuned_metrics = {}
if RUN_FINETUNE:
    display(Markdown("---"))
    display(Markdown("# 🔧 FINETUNING WITH QLoRA"))

    def sft_text(sample: dict) -> str:
        instr = sample.get("instruction","Analyze the sentiment of this Singapore property comment:")
        text  = sample.get("input","")
        out   = sample.get("output","")
        user  = f"{SYSTEM_MSG}\n\n{instr}\n\nInput: {text}\n\nReturn only the Output block."
        return f"<|prompt|>{user}</s><|answer|>{out}"

    train_ds = Dataset.from_list([{"text": sft_text(x)} for x in train_data])
    val_ds   = Dataset.from_list([{"text": sft_text(x)} for x in val_data])

    def tok_fn(batch):
        t = tokenizer(batch["text"], max_length=MAX_LEN, truncation=True, padding="max_length")
        t["labels"] = t["input_ids"].copy()
        return t

    train_ds = train_ds.map(tok_fn, batched=True, remove_columns=["text"])
    val_ds   = val_ds.map(tok_fn,   batched=True, remove_columns=["text"])

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
    )
    base_model.config.use_cache = False
    base_model = prepare_model_for_kbit_training(base_model)

    lora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(base_model, lora_cfg)

    bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    args = TrainingArguments(
        output_dir=str(FINETUNED_DIR),
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16 if not QUICK_TEST else 4,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        weight_decay=0.01,

        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=2,

        logging_steps=5,
        logging_dir=str(RESULTS_D2 / "tensorboard"),
        report_to=["tensorboard"],

        bf16=bf16_ok,
        fp16=not bf16_ok,

        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        gradient_checkpointing=True,
        remove_unused_columns=False,

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )
    early_stop = EarlyStoppingCallback(early_stopping_patience=2 if QUICK_TEST else 3)

    training_config = {
        "quick_test_mode": QUICK_TEST,
        "model_id": MODEL_ID,
        "train_samples": len(train_data),
        "val_samples": len(val_data),
        "test_samples": len(test_data),
        "epochs": EPOCHS,
        "max_len": MAX_LEN,
        "lr": args.learning_rate,
    }
    json.dump(training_config, open(RESULTS_D2 / "training_config.json", "w"), indent=2)

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        callbacks=[early_stop],
        tokenizer=tokenizer
    )

    display(Markdown("⏳ Training in progress..."))

    import datetime
    train_start_datetime = datetime.datetime.now().isoformat()
    train_result = trainer.train()
    train_end_datetime = datetime.datetime.now().isoformat()
    train_duration = (datetime.datetime.fromisoformat(train_end_datetime) -
                     datetime.datetime.fromisoformat(train_start_datetime)).total_seconds()

---

# 🔧 FINETUNING WITH QLoRA

Map:   0%|          | 0/2825 [00:00<?, ? examples/s]

Map:   0%|          | 0/314 [00:00<?, ? examples/s]

⏳ Training in progress...

Step,Training Loss,Validation Loss
50,7.119100,7.115824
100,7.397900,7.106957
150,7.347600,7.102293
200,6.882500,7.098671
250,7.352800,7.096321
300,7.191200,7.094069
350,7.268500,7.092199
400,7.276800,7.094648
450,6.942200,7.093674
500,7.279700,7.093626


In [ ]:
    # ========== SAVE COMPREHENSIVE TRAINING LOGS ==========
    display(Markdown("💾 Saving training logs..."))

    # 1) Save trainer state (complete training state)
    trainer_state_dict = {
        "global_step": trainer.state.global_step,
        "epoch": trainer.state.epoch,
        "total_flos": trainer.state.total_flos,
        "best_metric": trainer.state.best_metric,
        "best_model_checkpoint": trainer.state.best_model_checkpoint,
        "is_local_process_zero": trainer.state.is_local_process_zero,
        "is_world_process_zero": trainer.state.is_world_process_zero,
    }
    json.dump(trainer_state_dict, open(RESULTS_D2 / "trainer_state.json", "w"), indent=2)

    # 2) Save detailed training metrics
    training_metrics = {
        "start_time": train_start_datetime,
        "end_time": train_end_datetime,
        "total_duration_seconds": train_duration,
        "total_duration_formatted": f"{int(train_duration//3600)}h {int((train_duration%3600)//60)}m {int(train_duration%60)}s",
        "train_runtime": train_result.metrics.get("train_runtime", 0),
        "train_samples_per_second": train_result.metrics.get("train_samples_per_second", 0),
        "train_steps_per_second": train_result.metrics.get("train_steps_per_second", 0),
        "total_train_steps": trainer.state.global_step,
        "train_loss": train_result.metrics.get("train_loss", 0),
        "epoch": train_result.metrics.get("epoch", 0),
    }
    json.dump(training_metrics, open(RESULTS_D2 / "training_metrics.json", "w"), indent=2)

    # 3) Save raw training log history
    history = trainer.state.log_history
    json.dump(history, open(RESULTS_D2 / "training_log_history.json", "w"), indent=2)

    # 4) Create human-readable training log file
    log_file_path = RESULTS_D2 / "training.log"
    with open(log_file_path, "w") as log_file:
        log_file.write("="*80 + "\n")
        log_file.write("PROPINSIGHT DANUBE2 FINETUNING - TRAINING LOG\n")
        log_file.write("="*80 + "\n\n")

        log_file.write(f"Training Start: {train_start_datetime}\n")
        log_file.write(f"Training End:   {train_end_datetime}\n")
        log_file.write(f"Total Duration: {training_metrics['total_duration_formatted']}\n")
        log_file.write(f"Quick Test Mode: {QUICK_TEST}\n\n")

        log_file.write("-"*80 + "\n")
        log_file.write("CONFIGURATION\n")
        log_file.write("-"*80 + "\n")
        log_file.write(f"Model ID: {MODEL_ID}\n")
        log_file.write(f"Train Samples: {len(train_data)}\n")
        log_file.write(f"Val Samples: {len(val_data)}\n")
        log_file.write(f"Test Samples: {len(test_data)}\n")
        log_file.write(f"Epochs: {EPOCHS}\n")
        log_file.write(f"Max Length: {MAX_LEN}\n")
        log_file.write(f"Learning Rate: {args.learning_rate}\n")
        log_file.write(f"Batch Size: {args.per_device_train_batch_size}\n")
        log_file.write(f"Gradient Accumulation: {args.gradient_accumulation_steps}\n")
        log_file.write(f"Effective Batch Size: {args.per_device_train_batch_size * args.gradient_accumulation_steps}\n\n")

        log_file.write("-"*80 + "\n")
        log_file.write("LORA CONFIGURATION\n")
        log_file.write("-"*80 + "\n")
        log_file.write(f"LoRA r: {lora_cfg.r}\n")
        log_file.write(f"LoRA alpha: {lora_cfg.lora_alpha}\n")
        log_file.write(f"LoRA dropout: {lora_cfg.lora_dropout}\n")
        log_file.write(f"Target modules: {', '.join(lora_cfg.target_modules)}\n\n")

        log_file.write("-"*80 + "\n")
        log_file.write("TRAINING PROGRESS (STEP-BY-STEP)\n")
        log_file.write("-"*80 + "\n\n")

        for entry in history:
            step = entry.get("step", "N/A")
            epoch_val = entry.get("epoch", "N/A")

            if "loss" in entry:
                log_file.write(f"[Step {step:>5}] Epoch: {epoch_val:.2f} | Train Loss: {entry['loss']:.6f}")
                if "learning_rate" in entry:
                    log_file.write(f" | LR: {entry['learning_rate']:.2e}")
                log_file.write("\n")

            if "eval_loss" in entry:
                log_file.write(f"[Step {step:>5}] >>> EVAL <<< | Val Loss: {entry['eval_loss']:.6f}")
                if "eval_runtime" in entry:
                    log_file.write(f" | Runtime: {entry['eval_runtime']:.2f}s")
                log_file.write("\n")

        log_file.write("\n" + "-"*80 + "\n")
        log_file.write("TRAINING SUMMARY\n")
        log_file.write("-"*80 + "\n")
        log_file.write(f"Total Steps: {trainer.state.global_step}\n")
        log_file.write(f"Final Train Loss: {train_result.metrics.get('train_loss', 'N/A')}\n")
        log_file.write(f"Best Metric: {trainer.state.best_metric}\n")
        log_file.write(f"Best Checkpoint: {trainer.state.best_model_checkpoint}\n")
        log_file.write(f"Samples/Second: {training_metrics['train_samples_per_second']:.2f}\n")
        log_file.write(f"Steps/Second: {training_metrics['train_steps_per_second']:.2f}\n")

    print(f"✓ Training log saved to {log_file_path}")

    # 5) Create training curves visualization
    train_losses = [entry['loss'] for entry in history if 'loss' in entry]
    train_steps = [entry['step'] for entry in history if 'loss' in entry]
    eval_losses = [entry['eval_loss'] for entry in history if 'eval_loss' in entry]
    eval_steps = [entry['step'] for entry in history if 'eval_loss' in entry]
    learning_rates = [entry.get('learning_rate', 0) for entry in history if 'loss' in entry]

    # Create 4-panel visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Panel 1: Training Loss
    axes[0, 0].plot(train_steps, train_losses, label='Train Loss', color='#3498db', linewidth=2)
    axes[0, 0].set_xlabel('Step', fontsize=11, fontweight='bold')
    axes[0, 0].set_ylabel('Loss', fontsize=11, fontweight='bold')
    axes[0, 0].set_title('Training Loss over Steps', fontsize=12, fontweight='bold')
    axes[0, 0].legend(fontsize=10)
    axes[0, 0].grid(True, alpha=0.3, linestyle='--')

    # Panel 2: Validation Loss
    if eval_steps:
        axes[0, 1].plot(eval_steps, eval_losses, label='Val Loss', color='#e74c3c', linewidth=2, marker='o')
        axes[0, 1].set_xlabel('Step', fontsize=11, fontweight='bold')
        axes[0, 1].set_ylabel('Loss', fontsize=11, fontweight='bold')
        axes[0, 1].set_title('Validation Loss over Steps', fontsize=12, fontweight='bold')
        axes[0, 1].legend(fontsize=10)
        axes[0, 1].grid(True, alpha=0.3, linestyle='--')

    # Panel 3: Combined Train & Val
    axes[1, 0].plot(train_steps, train_losses, label='Train Loss', color='#3498db', linewidth=2, alpha=0.7)
    if eval_steps:
        axes[1, 0].plot(eval_steps, eval_losses, label='Val Loss', color='#e74c3c', linewidth=2, marker='o')
    axes[1, 0].set_xlabel('Step', fontsize=11, fontweight='bold')
    axes[1, 0].set_ylabel('Loss', fontsize=11, fontweight='bold')
    axes[1, 0].set_title('Training & Validation Loss', fontsize=12, fontweight='bold')
    axes[1, 0].legend(fontsize=10)
    axes[1, 0].grid(True, alpha=0.3, linestyle='--')

    # Panel 4: Learning Rate
    axes[1, 1].plot(train_steps, learning_rates, label='Learning Rate', color='#27ae60', linewidth=2)
    axes[1, 1].set_xlabel('Step', fontsize=11, fontweight='bold')
    axes[1, 1].set_ylabel('Learning Rate', fontsize=11, fontweight='bold')
    axes[1, 1].set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
    axes[1, 1].legend(fontsize=10)
    axes[1, 1].grid(True, alpha=0.3, linestyle='--')
    axes[1, 1].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

    plt.suptitle('Training Curves Overview', fontsize=15, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(VIS_D2 / "training_curves.png", dpi=300, bbox_inches='tight')
    plt.close()

    # 6) Save training history data for further analysis
    training_history = {
        "train_steps": train_steps,
        "train_losses": train_losses,
        "eval_steps": eval_steps,
        "eval_losses": eval_losses,
        "learning_rates": learning_rates
    }
    json.dump(training_history, open(RESULTS_D2 / "training_history.json", "w"), indent=2)

    # 7) Create epoch-based validation loss plot
    epoch_eval_data = {}
    for entry in history:
        if 'eval_loss' in entry and 'epoch' in entry:
            epoch = entry['epoch']
            if epoch not in epoch_eval_data:
                epoch_eval_data[epoch] = entry['eval_loss']

    if epoch_eval_data:
        epochs = sorted(epoch_eval_data.keys())
        losses = [epoch_eval_data[e] for e in epochs]

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(epochs, losses, marker='o', linewidth=2, markersize=8, color='#e74c3c')
        ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
        ax.set_ylabel('Validation Loss', fontsize=12, fontweight='bold')
        ax.set_title('Validation Loss per Epoch', fontsize=13, fontweight='bold', pad=10)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # Add value labels
        for x, y in zip(epochs, losses):
            ax.text(x, y + max(losses)*0.02, f'{y:.4f}', ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plt.savefig(VIS_D2 / "training_epochs.png", dpi=300, bbox_inches='tight')
        plt.close()

    print(f"✓ Training visualizations saved to {VIS_D2}")
    # ========================================

    model.save_pretrained(FINETUNED_DIR)
    tokenizer.save_pretrained(FINETUNED_DIR)
    display(Markdown(f"✅ **Finetuned model saved to `{FINETUNED_DIR}`**"))

💾 Saving training logs...

✓ Training log saved to /content/drive/MyDrive/PropInsight/results/_Danube2/training.log
✓ Training visualizations saved to /content/drive/MyDrive/PropInsight/visualizations/_Danube2


✅ **Finetuned model saved to `/content/drive/MyDrive/PropInsight/models/finetuned/_Danube2`**

In [ ]:
# FINETUNED EVALUATION
display(Markdown("---"))
display(Markdown("# 🎯 FINETUNED MODEL EVALUATION"))

from peft import PeftModel
base_ft = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
finetuned_model = PeftModel.from_pretrained(base_ft, FINETUNED_DIR)
finetuned_model.eval()

finetuned_preds = []
for ex in tqdm(test_data, desc="Finetuned inference"):
    inp = ex['input']
    prompt = prompt_for_infer(inp)
    ids = tokenizer.encode(prompt, return_tensors="pt").to(finetuned_model.device)
    out_ids = finetuned_model.generate(
        ids, max_new_tokens=800, do_sample=False,
        pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id
    )
    raw = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    block = raw.split("<|answer|>")[-1].strip() if "<|answer|>" in raw else raw

    # Extract domain metrics
    singlish_score = check_singlish_coverage(
        inp, block, corpus_resources.get('singlish_lexicon')
    )
    entity_score = check_property_entities(
        inp, block,
        corpus_resources.get('property_glossary'),
        corpus_resources.get('regex_patterns')
    )

    finetuned_preds.append({
        "input": inp,
        "prediction": block,
        "overall_pred": pick_overall(block),
        "ground_truth": ex.get("output", ""),
        "singlish_score": singlish_score,
        "entity_score": entity_score,
    })

del finetuned_model, base_ft
torch.cuda.empty_cache()

json.dump(finetuned_preds, open(RESULTS_D2 / "finetuned_predictions.json", "w"), indent=2)

y_true_ft = [ex["output"].split("Overall Sentiment:")[-1].split("\n")[0].strip().lower()
             for ex in test_data]
y_pred_ft = [p["overall_pred"] for p in finetuned_preds]
labels_ft = sorted(set(y_true_ft + y_pred_ft))

# Standard metrics
finetuned_metrics = {
    "overall_sentiment_accuracy": float(accuracy_score(y_true_ft, y_pred_ft)),
    "macro_precision": float(precision_score(y_true_ft, y_pred_ft, labels=labels_ft, average='macro', zero_division=0)),
    "macro_recall": float(recall_score(y_true_ft, y_pred_ft, labels=labels_ft, average='macro', zero_division=0)),
    "macro_f1": float(f1_score(y_true_ft, y_pred_ft, labels=labels_ft, average='macro', zero_division=0)),
    "weighted_f1": float(f1_score(y_true_ft, y_pred_ft, labels=labels_ft, average='weighted', zero_division=0)),
}

# Domain-specific metrics
singlish_scores = [p['singlish_score'] for p in finetuned_preds]
entity_scores = [p['entity_score'] for p in finetuned_preds]

finetuned_metrics.update({
    "singlish_coverage": float(np.mean(singlish_scores)) if singlish_scores else 0.0,
    "entity_recognition": float(np.mean(entity_scores)) if entity_scores else 0.0,
})
finetuned_metrics["domain_avg"] = (
    finetuned_metrics["singlish_coverage"] + finetuned_metrics["entity_recognition"]
) / 2.0

# BLEU & ROUGE (with error handling)
try:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    from rouge_score import rouge_scorer
    import nltk
    nltk.download('punkt', quiet=True)
    from nltk.tokenize import word_tokenize

    rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    bleu_scores, rouge1_scores, rouge2_scores, rougeL_scores = [], [], [], []
    smoothing = SmoothingFunction().method1

    for pred in finetuned_preds:
        gt = pred['ground_truth']
        pr = pred['prediction']

        if gt and pr:
            try:
                # BLEU
                reference = [word_tokenize(gt.lower())]
                hypothesis = word_tokenize(pr.lower())
                bleu = sentence_bleu(reference, hypothesis, smoothing_function=smoothing)
                bleu_scores.append(bleu)

                # ROUGE
                rouge_result = rouge_scorer_obj.score(gt, pr)
                rouge1_scores.append(rouge_result['rouge1'].fmeasure)
                rouge2_scores.append(rouge_result['rouge2'].fmeasure)
                rougeL_scores.append(rouge_result['rougeL'].fmeasure)
            except:
                bleu_scores.append(0)
                rouge1_scores.append(0)
                rouge2_scores.append(0)
                rougeL_scores.append(0)

    # Add to metrics
    finetuned_metrics.update({
        "bleu_score": float(np.mean(bleu_scores)) if bleu_scores else 0.0,
        "rouge1_score": float(np.mean(rouge1_scores)) if rouge1_scores else 0.0,
        "rouge2_score": float(np.mean(rouge2_scores)) if rouge2_scores else 0.0,
        "rougeL_score": float(np.mean(rougeL_scores)) if rougeL_scores else 0.0,
        "labels": labels_ft,
    })
except ImportError:
    print("⚠️  NLTK/ROUGE packages not available. Skipping NLP metrics.")
    finetuned_metrics["labels"] = labels_ft

# Calculate per-label F1 scores
per_label_f1 = {}
for label in labels_ft:
    label_mask = [yt == label for yt in y_true_ft]
    if sum(label_mask) > 0:
        label_f1 = f1_score(
            [yt for yt, m in zip(y_true_ft, label_mask) if m],
            [yp for yp, m in zip(y_pred_ft, label_mask) if m],
            labels=[label],
            average='micro',
            zero_division=0
        )
        per_label_f1[label] = float(label_f1)

finetuned_metrics['per_label_f1'] = per_label_f1

json.dump(finetuned_metrics, open(RESULTS_D2 / "finetuned_metrics.json", "w"), indent=2)

# ========== NLP METRICS VISUALIZATION ==========
if 'bleu_score' in finetuned_metrics:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: NLP Metrics Bar Chart
    nlp_metrics = ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L']
    nlp_values = [
        finetuned_metrics['bleu_score'],
        finetuned_metrics['rouge1_score'],
        finetuned_metrics['rouge2_score'],
        finetuned_metrics['rougeL_score']
    ]

    bars = axes[0].bar(nlp_metrics, nlp_values, color=['#3498db', '#e74c3c', '#f39c12', '#9b59b6'],
                      alpha=0.8, edgecolor='black', linewidth=1.2)
    for bar in bars:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                    f'{height:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

    axes[0].set_ylabel('Score', fontsize=12, fontweight='bold')
    axes[0].set_title('NLP Generation Metrics (Finetuned Model)', fontsize=13, fontweight='bold', pad=10)
    axes[0].set_ylim(0, 1.1)
    axes[0].grid(True, alpha=0.3, axis='y', linestyle='--')
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)

    # Right: Per-Label F1 Scores
    if per_label_f1:
        sorted_labels = sorted(per_label_f1.items(), key=lambda x: x[1], reverse=True)[:15]  # Top 15
        label_names = [l[0] for l in sorted_labels]
        label_f1s = [l[1] for l in sorted_labels]

        bars = axes[1].barh(label_names, label_f1s, color='#27ae60', alpha=0.8, edgecolor='black', linewidth=1)
        for i, (bar, val) in enumerate(zip(bars, label_f1s)):
            axes[1].text(val + 0.02, i, f'{val:.3f}', va='center', fontsize=9, fontweight='bold')

        axes[1].set_xlabel('F1 Score', fontsize=12, fontweight='bold')
        axes[1].set_title('Per-Sentiment F1 Scores (Top 15)', fontsize=13, fontweight='bold', pad=10)
        axes[1].set_xlim(0, 1.1)
        axes[1].grid(True, alpha=0.3, axis='x', linestyle='--')
        axes[1].spines['top'].set_visible(False)
        axes[1].spines['right'].set_visible(False)
    else:
        axes[1].text(0.5, 0.5, "No per-label data", ha='center', va='center', fontsize=12)
        axes[1].set_title('Per-Sentiment F1 Scores', fontsize=13, fontweight='bold')
        axes[1].axis('off')

    plt.tight_layout()
    plt.savefig(VIS_D2 / "nlp_metrics.png", dpi=300, bbox_inches='tight')
    plt.close()

    display(Markdown(f"""
    **NLP Metrics (Finetuned):**
    - BLEU Score: {finetuned_metrics['bleu_score']:.4f}
    - ROUGE-1: {finetuned_metrics['rouge1_score']:.4f}
    - ROUGE-2: {finetuned_metrics['rouge2_score']:.4f}
    - ROUGE-L: {finetuned_metrics['rougeL_score']:.4f}
    """))

plot_sentiment_distribution(y_true_ft, y_pred_ft,
                           "Overall Sentiment Distribution — Finetuned (Danube2)",
                           VIS_D2 / "finetuned_sentiment_comparison.png",
                           colors=['#3498db', '#27ae60'])

plot_confusion_matrix(y_true_ft, y_pred_ft, labels_ft,
                     "Finetuned Confusion Matrix (Danube2)",
                     VIS_D2 / "finetuned_confusion_matrix.png",
                     cmap='Greens')

display(Markdown("✅ **Finetuned evaluation complete!**"))

---

# 🎯 FINETUNED MODEL EVALUATION

Finetuned inference:   0%|          | 0/554 [00:00<?, ?it/s]

⚠️  NLTK/ROUGE packages not available. Skipping NLP metrics.


✅ **Finetuned evaluation complete!**

In [ ]:
#Comparison
if RUN_BASELINE and RUN_FINETUNE:
    display(Markdown("---"))
    display(Markdown("# 📊 FINAL COMPARISON"))

    display_metrics_table(baseline_metrics, finetuned_metrics)

    comparison_metrics = {
        "baseline": baseline_metrics,
        "finetuned": finetuned_metrics,
        "improvements": {
            "accuracy_delta": finetuned_metrics["overall_sentiment_accuracy"] - baseline_metrics["overall_sentiment_accuracy"],
            "f1_delta": finetuned_metrics["macro_f1"] - baseline_metrics["macro_f1"],
            "precision_delta": finetuned_metrics["macro_precision"] - baseline_metrics["macro_precision"],
            "recall_delta": finetuned_metrics["macro_recall"] - baseline_metrics["macro_recall"],
            "singlish_coverage_delta": finetuned_metrics.get("singlish_coverage", 0) - baseline_metrics.get("singlish_coverage", 0),
            "entity_recognition_delta": finetuned_metrics.get("entity_recognition", 0) - baseline_metrics.get("entity_recognition", 0),
            "domain_avg_delta": finetuned_metrics.get("domain_avg", 0) - baseline_metrics.get("domain_avg", 0),
        }
    }
    json.dump(comparison_metrics, open(RESULTS_D2 / "comparison_metrics.json", "w"), indent=2)

    # ========== DOMAIN METRICS VISUALIZATION ==========
    plot_domain_metrics(
        baseline_metrics,
        finetuned_metrics,
        VIS_D2 / "domain_metrics_comparison.png"
    )
    # ========================================

    display(Markdown(f"""
    ### 📁 Results saved to:
    **Metrics & Predictions:**
    - `{RESULTS_D2 / "baseline_predictions.json"}`
    - `{RESULTS_D2 / "baseline_metrics.json"}` ⭐ (includes domain metrics)
    - `{RESULTS_D2 / "finetuned_predictions.json"}`
    - `{RESULTS_D2 / "finetuned_metrics.json"}` ⭐ (includes BLEU, ROUGE, domain metrics)
    - `{RESULTS_D2 / "comparison_metrics.json"}` ⭐ (includes domain improvements)

    **Training Logs:**
    - `{RESULTS_D2 / "training.log"}` ⭐ (human-readable)
    - `{RESULTS_D2 / "training_log_history.json"}` (complete history)
    - `{RESULTS_D2 / "training_metrics.json"}` (summary metrics)
    - `{RESULTS_D2 / "training_history.json"}` (loss curves data)
    - `{RESULTS_D2 / "trainer_state.json"}` (trainer state)
    - `{RESULTS_D2 / "training_config.json"}` (config)
    - `{RESULTS_D2 / "tensorboard/"}` (TensorBoard logs)

    **Visualizations:**
    - `{VIS_D2 / "training_curves.png"}` ⭐ (4-panel training overview)
    - `{VIS_D2 / "training_epochs.png"}` (validation loss per epoch)
    - `{VIS_D2 / "nlp_metrics.png"}` ⭐ (BLEU, ROUGE, per-label F1)
    - `{VIS_D2 / "domain_metrics_comparison.png"}` ⭐ (Singlish & Entity metrics)
    - `{VIS_D2 / "baseline_sentiment_comparison.png"}`
    - `{VIS_D2 / "baseline_confusion_matrix.png"}`
    - `{VIS_D2 / "finetuned_sentiment_comparison.png"}`
    - `{VIS_D2 / "finetuned_confusion_matrix.png"}`

    **Model:**
    - `{FINETUNED_DIR}` (LoRA adapters + tokenizer)

    **Domain Knowledge:**
    🌏 Fine-tuning improved Singlish coverage by **{comparison_metrics['improvements']['singlish_coverage_delta']:+.3f}**
    🏢 Fine-tuning improved entity recognition by **{comparison_metrics['improvements']['entity_recognition_delta']:+.3f}**
    📊 Overall domain knowledge improved by **{comparison_metrics['improvements']['domain_avg_delta']:+.3f}**
    """))

if QUICK_TEST:
    display(HTML("""
    <div style="background-color: #4CAF50; color: white; padding: 20px; border-radius: 8px; margin: 20px 0; text-align: center;">
        <h2 style="margin: 0;">✅ QUICK TEST COMPLETE!</h2>
        <p style="margin: 10px 0 0 0;">Pipeline verified successfully. Set <code>QUICK_TEST = False</code> for full training.</p>
    </div>
    """))
else:
    display(Markdown("---"))
    display(Markdown("# 🎉 ALL TASKS COMPLETE!"))

print("\n" + "="*60)
print("✅ DONE!")
print("="*60)



---

# 📊 FINAL COMPARISON

Metric,Baseline,Finetuned,Δ (Improvement)
Overall Sentiment Accuracy,0.2942,0.8375,+0.5433
Macro Precision,0.0429,0.7866,+0.7437
Macro Recall,0.0515,0.7539,+0.7024
Macro F1,0.0333,0.7685,+0.7352
Weighted F1,0.2604,0.8353,+0.5749
Singlish Coverage,1.0000,1.0000,+0.0000
Entity Recognition,0.6697,0.6480,-0.0217
Domain Avg,0.8348,0.8240,-0.0108


✓ Domain metrics visualization saved to /content/drive/MyDrive/PropInsight/visualizations/_Danube2/domain_metrics_comparison.png



    ### 📁 Results saved to:
    **Metrics & Predictions:**
    - `/content/drive/MyDrive/PropInsight/results/_Danube2/baseline_predictions.json`
    - `/content/drive/MyDrive/PropInsight/results/_Danube2/baseline_metrics.json` ⭐ (includes domain metrics)
    - `/content/drive/MyDrive/PropInsight/results/_Danube2/finetuned_predictions.json`
    - `/content/drive/MyDrive/PropInsight/results/_Danube2/finetuned_metrics.json` ⭐ (includes BLEU, ROUGE, domain metrics)
    - `/content/drive/MyDrive/PropInsight/results/_Danube2/comparison_metrics.json` ⭐ (includes domain improvements)
    
    **Training Logs:**
    - `/content/drive/MyDrive/PropInsight/results/_Danube2/training.log` ⭐ (human-readable)
    - `/content/drive/MyDrive/PropInsight/results/_Danube2/training_log_history.json` (complete history)
    - `/content/drive/MyDrive/PropInsight/results/_Danube2/training_metrics.json` (summary metrics)
    - `/content/drive/MyDrive/PropInsight/results/_Danube2/training_history.json` (loss curves data)
    - `/content/drive/MyDrive/PropInsight/results/_Danube2/trainer_state.json` (trainer state)
    - `/content/drive/MyDrive/PropInsight/results/_Danube2/training_config.json` (config)
    - `/content/drive/MyDrive/PropInsight/results/_Danube2/tensorboard` (TensorBoard logs)
    
    **Visualizations:**
    - `/content/drive/MyDrive/PropInsight/visualizations/_Danube2/training_curves.png` ⭐ (4-panel training overview)
    - `/content/drive/MyDrive/PropInsight/visualizations/_Danube2/training_epochs.png` (validation loss per epoch)
    - `/content/drive/MyDrive/PropInsight/visualizations/_Danube2/nlp_metrics.png` ⭐ (BLEU, ROUGE, per-label F1)
    - `/content/drive/MyDrive/PropInsight/visualizations/_Danube2/domain_metrics_comparison.png` ⭐ (Singlish & Entity metrics)
    - `/content/drive/MyDrive/PropInsight/visualizations/_Danube2/baseline_sentiment_comparison.png`
    - `/content/drive/MyDrive/PropInsight/visualizations/_Danube2/baseline_confusion_matrix.png`
    - `/content/drive/MyDrive/PropInsight/visualizations/_Danube2/finetuned_sentiment_comparison.png`
    - `/content/drive/MyDrive/PropInsight/visualizations/_Danube2/finetuned_confusion_matrix.png`
    
    **Model:**
    - `/content/drive/MyDrive/PropInsight/models/finetuned/_Danube2` (LoRA adapters + tokenizer)
    
    **Domain Knowledge:**
    🌏 Fine-tuning improved Singlish coverage by **+0.000**
    🏢 Fine-tuning improved entity recognition by **-0.022**
    📊 Overall domain knowledge improved by **-0.011**
    

---

# 🎉 ALL TASKS COMPLETE!


✅ DONE!
